# Results & Cost Analysis

Companion analysis for the **startup-book** generator (guidelines §9 research,
§11 costs). It reads the project configuration and visualises (1) the estimated
API run cost as a function of token volume, and (2) a unit-economics sensitivity
tied to the book's *Unit Economics* chapter.

Run from the repo root with the project venv:
`uv run jupyter nbconvert --to notebook --execute notebooks/results_analysis.ipynb`

In [1]:
import matplotlib
import numpy as np

matplotlib.use("Agg")
import matplotlib.pyplot as plt  # noqa: E402

from startup_book.constants import RESULTS_DIR
from startup_book.shared.config import ConfigManager

# Write to the canonical repo-root results/ regardless of the kernel's cwd.
results_dir = RESULTS_DIR
results_dir.mkdir(exist_ok=True)
cfg = ConfigManager()
in_rate, out_rate = cfg.cost_rates()
model = cfg.model()
print(f"model={model}  input=${in_rate}/1M  output=${out_rate}/1M")

model=gpt-4o-mini  input=$0.15/1M  output=$0.6/1M


## 1. Estimated run cost vs token volume

With an assumed 30% prompt / 70% completion split, the cost of one book run is
$$\text{cost} = \frac{0.3\,T}{10^6}\,r_{in} + \frac{0.7\,T}{10^6}\,r_{out},$$
where $T$ is the total token count and $r_{in}, r_{out}$ are the per-million
input/output prices from `config/setup.json`.

In [2]:
totals = np.linspace(0, 60_000, 200)
cost = (0.3 * totals / 1e6) * in_rate + (0.7 * totals / 1e6) * out_rate

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(totals / 1000, cost, color="#1f3b73", lw=2)
ax.set_xlabel("Total tokens (thousands)")
ax.set_ylabel("Estimated cost (USD)")
ax.set_title(f"Estimated run cost — {model}")
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(results_dir / "cost_vs_tokens.pdf")
approx = (0.3 * 30000 / 1e6) * in_rate + (0.7 * 30000 / 1e6) * out_rate
print(f"~30k-token book run ≈ ${approx:.4f}")

~30k-token book run ≈ $0.0140


## 2. Unit-economics sensitivity (LTV vs churn)

From the book's economics chapter, the customer lifetime value is
$$\mathrm{LTV} = \frac{\overline{\mathrm{ARPA}}\cdot m_{\text{gross}}}{c_{\text{churn}}}.$$
Here we hold $\overline{\mathrm{ARPA}}=\$50$ and $m_{\text{gross}}=0.8$ and sweep
monthly churn.

In [3]:
churn = np.linspace(0.01, 0.10, 100)
ltv = 50 * 0.8 / churn

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(churn * 100, ltv, color="#2e7d57", lw=2)
ax.axhline(360, color="#c0392b", ls="--", label="3x of a $120 CAC")
ax.set_xlabel("Monthly churn (%)")
ax.set_ylabel("LTV (USD)")
ax.set_title("LTV sensitivity to churn (ARPA=$50, margin=80%)")
ax.legend(frameon=False)
ax.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(results_dir / "ltv_sensitivity.pdf")
print(f"LTV at 2% churn = ${50 * 0.8 / 0.02:.0f}")

LTV at 2% churn = $2000


**Takeaways.**
- **Cost is tiny and linear.** A full book on `gpt-4o-mini` costs ~1.4¢; the
  breakdown (§3) shows it is ~13–17× cheaper (blended) than the larger GPT-4o /
  4.1 models, and the scaling projection (§4) confirms cost grows linearly —
  1000 books ≈ \$14. The marginal cost per book is the number to manage.
- **Length is the lever.** The word-target study (§5) ties the per-chapter word
  budget directly to tokens, page count and cost — the knob to turn for a target
  page count, since prompt/output length dominates spend.
- **Unit economics are churn-dominated** (§2): halving churn doubles LTV, which
  is why retention (Product–Market Fit) precedes growth spend.

All figures/tables are written under `results/` so the analysis is reproducible
from `config/setup.json` alone.

In [4]:
import csv

# Illustrative provider list prices (USD per 1M tokens) — edit to match your
# provider and date. The configured model's row is overwritten with the exact
# values from config/setup.json so it is always accurate.
PRICES = {
    "gpt-4o-mini": (0.15, 0.60),
    "gpt-4.1-mini": (0.40, 1.60),
    "gpt-4o": (2.50, 10.00),
    "gpt-4.1": (2.00, 8.00),
}
PRICES[model] = (in_rate, out_rate)

SPLIT_IN, SPLIT_OUT = 0.30, 0.70  # assumed prompt/completion mix
BOOK_TOKENS_REF = 30_000

rows = []
for name, (ri, ro) in sorted(PRICES.items()):
    blended = SPLIT_IN * ri + SPLIT_OUT * ro
    per_book = (SPLIT_IN * BOOK_TOKENS_REF / 1e6) * ri + (SPLIT_OUT * BOOK_TOKENS_REF / 1e6) * ro
    rows.append((name, ri, ro, blended, per_book, 1.0 / per_book if per_book else float("inf")))

csv_path = results_dir / "model_cost_comparison.csv"
with csv_path.open("w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["model", "in_per_1m", "out_per_1m", "blended_per_1m", "usd_per_book_30k", "books_per_usd"])
    w.writerows(rows)

hdr = f"{'model':<14}{'$/1M in':>9}{'$/1M out':>10}{'blended':>10}{'$/book':>10}{'books/$':>10}"
print(hdr)
print("-" * len(hdr))
for name, ri, ro, bl, pb, bpu in rows:
    mark = "  <- configured" if name == model else ""
    print(f"{name:<14}{ri:>9.2f}{ro:>10.2f}{bl:>10.2f}{pb:>10.4f}{bpu:>10.0f}{mark}")
print(f"\nsaved {csv_path}")

model           $/1M in  $/1M out   blended    $/book   books/$
---------------------------------------------------------------
gpt-4.1            2.00      8.00      6.20    0.1860         5
gpt-4.1-mini       0.40      1.60      1.24    0.0372        27
gpt-4o             2.50     10.00      7.75    0.2325         4
gpt-4o-mini        0.15      0.60      0.46    0.0140        72  <- configured

saved /mnt/c/Orch/HW3/results/model_cost_comparison.csv


## 4. Scaling projection — cost to mass-produce N books

The course is about the *mass production* of agents, so the operative question
is the marginal cost per book and how a fleet scales. Cost is linear in book
count; the log–log plot makes the per-book constant explicit and the table is
written to `results/cost_scaling.csv`.

In [ ]:
BOOK_TOKENS = 30_000  # representative tokens for one full 8-chapter book run
per_book = (SPLIT_IN * BOOK_TOKENS / 1e6) * in_rate + (SPLIT_OUT * BOOK_TOKENS / 1e6) * out_rate

n_books = np.array([1, 10, 50, 100, 500, 1000])
fleet_cost = n_books * per_book

fig, ax = plt.subplots(figsize=(6, 3.5))
ax.plot(n_books, fleet_cost, "o-", color="#1f3b73", lw=2)
ax.set_xscale("log")
ax.set_yscale("log")
ax.set_xlabel("Number of books (log)")
ax.set_ylabel("Projected API cost USD (log)")
ax.set_title(f"Cost to mass-produce N books — {model} (~{BOOK_TOKENS // 1000}k tok/book)")
ax.grid(alpha=0.3, which="both")
fig.tight_layout()
fig.savefig(results_dir / "cost_scaling.pdf")

with (results_dir / "cost_scaling.csv").open("w", newline="") as fh:
    w = csv.writer(fh)
    w.writerow(["n_books", "projected_cost_usd"])
    for n, c in zip(n_books, fleet_cost, strict=True):
        w.writerow([int(n), round(float(c), 4)])

print(f"per book ≈ ${per_book:.4f}")
print(f"100 books ≈ ${per_book * 100:.2f}   ·   1000 books ≈ ${per_book * 1000:.2f}")

## 5. Parameter study — chapter word-target → tokens, pages, cost

The dominant cost/length lever is how long each chapter is. Holding the
8-chapter outline fixed, we sweep the per-chapter **word target** and project
total tokens, page count and cost from simple heuristics (~1.33 tokens/word for
mixed Hebrew/English, a fixed prompt overhead per chapter, ~450 words/page).
This is the knob to turn when targeting a specific page budget.

In [6]:
# Heuristics: ~1.33 tokens/word for mixed Hebrew/English prose, a fixed prompt
# overhead per chapter (research + instructions), ~450 words/page in this layout.
TOK_PER_WORD = 1.33
PROMPT_OVERHEAD = 1200
WORDS_PER_PAGE = 450
n_chapters = len(cfg.chapter_specs())

word_target = np.linspace(300, 1500, 100)
completion_tok = word_target * TOK_PER_WORD * n_chapters
prompt_tok = PROMPT_OVERHEAD * n_chapters + 0.5 * completion_tok  # context grows with drafts
total_tok = prompt_tok + completion_tok
pages = word_target * n_chapters / WORDS_PER_PAGE
cost = prompt_tok / 1e6 * in_rate + completion_tok / 1e6 * out_rate

fig, ax1 = plt.subplots(figsize=(6.5, 3.8))
ax1.plot(word_target, total_tok / 1000, color="#1f3b73", lw=2)
ax1.set_xlabel("Words per chapter (target)")
ax1.set_ylabel("Total tokens (thousands)", color="#1f3b73")
ax1.tick_params(axis="y", labelcolor="#1f3b73")
ax2 = ax1.twinx()
ax2.plot(word_target, cost, color="#c0392b", lw=2, ls="--")
ax2.set_ylabel("Estimated cost (USD)", color="#c0392b")
ax2.tick_params(axis="y", labelcolor="#c0392b")
ax1.set_title(f"Word-target study — {n_chapters} chapters, {model}")
ax1.grid(alpha=0.3)
fig.tight_layout()
fig.savefig(results_dir / "word_target_study.pdf")

print(f"{'words/ch':>9}{'pages':>7}{'k tokens':>10}{'cost $':>9}")
for wt in (400, 700, 1000, 1300):
    ct = wt * TOK_PER_WORD * n_chapters
    pt = PROMPT_OVERHEAD * n_chapters + 0.5 * ct
    c = pt / 1e6 * in_rate + ct / 1e6 * out_rate
    print(f"{wt:>9}{wt * n_chapters / WORDS_PER_PAGE:>7.1f}{(pt + ct) / 1000:>10.1f}{c:>9.4f}")

 words/ch  pages  k tokens   cost $
      400    7.1      16.0   0.0043
      700   12.4      20.8   0.0065
     1000   17.8      25.6   0.0086
     1300   23.1      30.3   0.0108


**Takeaways.** The book run is inexpensive (cents) on `gpt-4o-mini`; cost scales
linearly with tokens, so the main lever is prompt/output length. Unit economics
are dominated by churn: halving churn doubles LTV, which is why retention
(§Product–Market Fit) precedes growth spend.